In [1]:
import requests
import frontmatter

print("Setup working 🚀")

Setup working 🚀


In [2]:
import io
import zipfile
import requests
import frontmatter

def read_repo_data(repo_owner, repo_name):
    prefix = "https://codeload.github.com"
    url = f"{prefix}/{repo_owner}/{repo_name}/zip/refs/heads/main"

    print(f"Downloading {repo_owner}/{repo_name}...")
    resp = requests.get(url)

    if resp.status_code != 200:
        raise Exception(f"Download failed: {resp.status_code}")

    repository_data = []

    zf = zipfile.ZipFile(io.BytesIO(resp.content))

    for file_info in zf.infolist():
        filename = file_info.filename.lower()

        if not (filename.endswith(".md") or filename.endswith(".mdx")):
            continue

        try:
            with zf.open(file_info) as f:
                content = f.read().decode("utf-8", errors="ignore")
                post = frontmatter.loads(content)

                data = post.to_dict()
                data["filename"] = filename

                repository_data.append(data)

        except Exception as e:
            print(f"Error processing {filename}: {e}")

    zf.close()
    return repository_data

In [3]:
dtc_faq = read_repo_data("DataTalksClub", "faq")

print("Total documents:", len(dtc_faq))
print(dtc_faq[0])

Total documents: 1285
{'description': 'Review and process open FAQ PRs', 'content': 'Go through all open pull requests one by one. For each PR:\n\n## 1. Show Details\n- PR number and title\n- Course and section (from PR body)\n- Related issue number\n- **ALWAYS check sort_order**: List files in target section (`ls _questions/<course>/<section>/`) to find highest number, verify PR uses next sequential\n- Full diff (use `gh pr diff <number>`)\n\n## 2. Check Against These Rules\n\n### Section Placement\n- **Kestra questions** → must be in `module-2` (workflow orchestration), NOT `general` or `module-1`\n- **Docker + Kestra** → still `module-2` (Kestra is primary topic)\n- **Docker-only** (pgAdmin, Postgres, etc.) → `module-1`\n\n### Relevance (Close If)\n- Basic Linux/SQL tutorials (vim installation, SQL JOIN concepts, etc.)\n- Generic programming not tied to course content\n- Already exists in DE zoomcamp when proposed for ML zoomcamp\n\n### Duplicates (Check Before Creating)\n- Search e

In [4]:
docs = read_repo_data("evidentlyai", "docs")

print(len(docs))

95


In [9]:
# 🚀 Pipeline Design (Day 1)

# Instead of one big function, we split into:
# 1. Download repo 📥
#    - Fetch GitHub ZIP using requests
#
# 2. Extract files 📂
#    - Use zipfile to read archive
#    - Filter .md / .mdx files
#
# 3. Parse content 🧠
#    - Use frontmatter to extract metadata + content
#
# Benefits:
# - Cleaner code
# - Easier debugging
# - Reusable components
# - Production-ready structure

In [5]:
import io
import zipfile
import requests
import frontmatter


def download_repo_zip(repo_owner, repo_name):
    url = f"https://codeload.github.com/{repo_owner}/{repo_name}/zip/refs/heads/main"
    response = requests.get(url)

    if response.status_code != 200:
        raise Exception(f"Failed to download repo: {response.status_code}")

    return io.BytesIO(response.content)


def extract_markdown_files(zip_bytes):
    zf = zipfile.ZipFile(zip_bytes)
    
    markdown_files = []

    for file_info in zf.infolist():
        filename = file_info.filename.lower()

        if filename.endswith(".md") or filename.endswith(".mdx"):
            markdown_files.append(file_info)

    return zf, markdown_files


def parse_markdown_files(zf, markdown_files):
    data = []

    for file_info in markdown_files:
        try:
            with zf.open(file_info) as f:
                content = f.read().decode("utf-8", errors="ignore")
                post = frontmatter.loads(content)

                item = {
                    "filename": file_info.filename,
                    "metadata": post.metadata,
                    "content": post.content
                }

                data.append(item)

        except Exception as e:
            print(f"Error: {file_info.filename} -> {e}")

    return data


def load_github_repo(repo_owner, repo_name):
    zip_bytes = download_repo_zip(repo_owner, repo_name)
    zf, markdown_files = extract_markdown_files(zip_bytes)
    parsed_data = parse_markdown_files(zf, markdown_files)

    return parsed_data

In [6]:
data = load_github_repo("DataTalksClub", "faq")

print("Total files:", len(data))
print(data[0])

Total files: 1285
{'filename': 'faq-main/.claude/commands/pr.md', 'metadata': {'description': 'Review and process open FAQ PRs'}, 'content': 'Go through all open pull requests one by one. For each PR:\n\n## 1. Show Details\n- PR number and title\n- Course and section (from PR body)\n- Related issue number\n- **ALWAYS check sort_order**: List files in target section (`ls _questions/<course>/<section>/`) to find highest number, verify PR uses next sequential\n- Full diff (use `gh pr diff <number>`)\n\n## 2. Check Against These Rules\n\n### Section Placement\n- **Kestra questions** → must be in `module-2` (workflow orchestration), NOT `general` or `module-1`\n- **Docker + Kestra** → still `module-2` (Kestra is primary topic)\n- **Docker-only** (pgAdmin, Postgres, etc.) → `module-1`\n\n### Relevance (Close If)\n- Basic Linux/SQL tutorials (vim installation, SQL JOIN concepts, etc.)\n- Generic programming not tied to course content\n- Already exists in DE zoomcamp when proposed for ML zoomc